In [1]:
# Required with some rust libraries.
from multiprocessing import set_start_method
set_start_method('spawn', force=True)

from pathlib import Path
import pandas as pd
import numpy as np

from data import BasinDataLake

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs")

root_dir = save_dir / 'datalakes' / 'training'
store = BasinDataLake(root_dir)

zarr_path = '/scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr2/20251004'

In [3]:
store.compute_and_store_stats(zarr_path, overwrite=True)

Dask client started. Dashboard at: http://127.0.0.1:8787/status


Basins:   0%|          | 1/643 [00:11<2:06:07, 11.79s/it]/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/dask/array/reductions.py:324: RuntimeWarning: All-NaN slice encountered
  return np.nanmax(x_chunk, axis=axis, keepdims=keepdims)
/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/dask/array/reductions.py:291: RuntimeWarning: All-NaN slice encountered
  return np.nanmin(x_chunk, axis=axis, keepdims=keepdims)
Basins: 100%|██████████| 643/643 [3:08:11<00:00, 17.56s/it]  


In [2]:
store.export_to_zarr(zarr_path, workers=8, start_date='2000-01-01', end_date='2024-12-31')

Processing 643 basins.


Processing basins: 100%|██████████| 643/643 [2:17:08<00:00, 12.80s/it, completed=643, basin=USGS-15565447]       


In [6]:
import xarray as xr

ds = xr.open_dataset(Path(zarr_path) / 'ABOM-320099010', engine='zarr')
ds

<xarray.Dataset> Size: 29MB
Dimensions:           (date: 3653, subbasin: 17)
Coordinates:
  * date              (date) datetime64[ns] 29kB 2015-01-01 ... 2024-12-31
  * subbasin          (subbasin) object 136B '56167339' ... 'ABOM-321906010'
Data variables: (12/58)
    area_det_u_river  (date, subbasin) float64 497kB ...
    area_detct_river  (date, subbasin) float64 497kB ...
    area_tot_u_river  (date, subbasin) float64 497kB ...
    area_total_river  (date, subbasin) float64 497kB ...
    area_wse_river    (date, subbasin) float64 497kB ...
    d2m_mean          (date, subbasin) float64 497kB ...
    ...                ...
    wse_r_u_river     (date, subbasin) float64 497kB ...
    wse_river         (date, subbasin) float64 497kB ...
    wse_u_river       (date, subbasin) float64 497kB ...
    xovr_cal_c_river  (date, subbasin) float64 497kB ...
    xovr_cal_q_river  (date, subbasin) float64 497kB ...
    xtrk_dist_river   (date, subbasin) float64 497kB ...

In [7]:
ds

<xarray.Dataset> Size: 29MB
Dimensions:           (date: 3653, subbasin: 17)
Coordinates:
  * date              (date) datetime64[ns] 29kB 2015-01-01 ... 2024-12-31
  * subbasin          (subbasin) object 136B '56167339' ... 'ABOM-321906010'
Data variables: (12/58)
    area_det_u_river  (date, subbasin) float64 497kB ...
    area_detct_river  (date, subbasin) float64 497kB ...
    area_tot_u_river  (date, subbasin) float64 497kB ...
    area_total_river  (date, subbasin) float64 497kB ...
    area_wse_river    (date, subbasin) float64 497kB ...
    d2m_mean          (date, subbasin) float64 497kB ...
    ...                ...
    wse_r_u_river     (date, subbasin) float64 497kB ...
    wse_river         (date, subbasin) float64 497kB ...
    wse_u_river       (date, subbasin) float64 497kB ...
    xovr_cal_c_river  (date, subbasin) float64 497kB ...
    xovr_cal_q_river  (date, subbasin) float64 497kB ...
    xtrk_dist_river   (date, subbasin) float64 497kB ...

In [ ]:
zarr_path = Path('/scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr2')
zarr_group_path = Path(zarr_path) / basin

In [ ]:
store.export_to_zarr(
    zarr_path, 
    workers=2,
    sources=['era5'], 
    start_date='1980-01-01', 
    end_date="2024-12-31", 
)

In [ ]:
basins = pd.read_csv("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph/basin_lists/proto_all.txt", header=None)
basins = basins[0].to_list()[0:10]
sources = ['era5']

In [ ]:
store.export_to_zarr(zarr_path, workers=6, start_date='1980-01-01', end_date="2024-12-31")

In [ ]:
from tqdm import tqdm
import xarray as xr

root_path = Path(zarr_path)
print(f"🔍 Starting validation of Zarr store at: {root_path}\n")

# Get all subdirectories in the root path, assuming they are sources
# source_dirs = [d for d in root_path.iterdir() if d.is_dir()]
source_dirs = [root_path / 'era5']

corrupted_paths = []
# Set up the outer progress bar for sources
for source_path in source_dirs:
    source_name = source_path.name
    
    basin_dirs = [d for d in source_path.iterdir() if d.is_dir()]
    
    # Set up the inner progress bar for basins within the current source
    for basin_path in tqdm(basin_dirs):
        try:
            # The core operation: try to open the Zarr group.
            # The 'with' statement ensures the store is properly closed.
            with xr.open_zarr(basin_path, consolidated=False):
                pass # If this succeeds, the group is valid.

        except ValueError as e:
            # This is the specific error we are looking for!
            if "conflicting sizes for dimension" in str(e):
                tqdm.write(f"❌ CORRUPTED: {basin_path.relative_to(root_path)}")
                corrupted_paths.append(basin_path)
            else:
                # It's a different kind of ValueError
                tqdm.write(f"❗️ OTHER ERROR (ValueError): {basin_path.relative_to(root_path)} -> {e}")

        except Exception as e:
            # Catch any other error (e.g., not a valid Zarr store, permissions issue)
            tqdm.write(f"❗️ OTHER ERROR ({type(e).__name__}): {basin_path.relative_to(root_path)} -> {e}")

print("\n" + "="*50)
print("✅ Validation Complete")
print("="*50)

if corrupted_paths:
    print(f"\nFound {len(corrupted_paths)} corrupted Zarr groups that need regeneration:")
    for path in corrupted_paths:
        print(f"  - {path}")
else:
    print("\n🎉 No corrupted Zarr groups found!")



In [ ]:
import shutil
for p in corrupted_paths:
    print(f"Removing {p}")
    shutil.rmtree(p)

In [ ]:
start_date='1980-01-01'
end_date="2024-12-31"
year_step = 50

date_range = pd.date_range(start_date, end_date, freq=f"{year_step}YE", inclusive="both")

# Ensure the last date is exactly end_date
if date_range[-1] != pd.Timestamp(end_date):
    date_range = date_range.append(pd.DatetimeIndex([end_date]))

for start, end in zip(date_range[:-1], date_range[1:]):
    print(f"{start} - {end}")

In [ ]:
import xarray as xr

ds = xr.open_dataset(
    "/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr/era5/UKEA-520894",
    engine="zarr",
)

ds

In [ ]:
import numpy as np

np.isnan(ds['e_mean'].values).any()

In [ ]:
list(ds.data_vars)

In [ ]:
time_gaps = {
    column: ds[[column]].to_array().isnull().any().item()
    for column in list(ds.data_vars)
}
time_gaps

In [ ]:
ds

In [ ]:
ds.isnull().any().any().values()

In [ ]:
p = Path(zarr_path)
groups = [d.name for d in p.iterdir() if d.is_dir()]
print(f"Discovered groups: {groups}")
group_datasets = [xr.open_zarr(p / g) for g in groups]
ds = xr.merge(group_datasets)
ds


In [ ]:
group_datasets[1]

In [ ]:
import xarray as xr 

zarr_path = '/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr'
ds_gauge = xr.open_zarr(zarr_path, group='era5')
ds_gauge

In [ ]:
# Example: Select a 15-day period 
start_slice = '1990-06-01'
end_slice = '1990-06-15'

# .sel() is the standard way to perform label-based selection
data_slice = ds_gauge.sel(
    date=slice(start_slice, end_slice),
)
data_slice

In [ ]:
data_slice.load()

In [ ]:
tmp.index.get_level_values('date').tz_localize(None)

In [ ]:
def read_file(fp):
    with open(fp, "r") as file:
        basin_list = file.readlines()
        basin_list = [basin.strip() for basin in basin_list]
    return basin_list

list_dir = save_dir.parent / 'multigraph' / 'basin_lists'
train_basins = read_file(list_dir / 'proto_train.txt')
test_basins = read_file(list_dir / 'proto_test.txt')

In [ ]:
static_df = store.read_static()


subbasins = (
    static_df.loc[static_df["basin"] == basin, "subbasin"]
    .astype(str)
    .unique()
    .tolist()
)
subbasins = sorted(subbasins)
subbasins

In [ ]:
df = tmp
value_cols = [c for c in df.columns if c not in ["basin", "year"]]
value_cols

In [ ]:
import xarray as xr
ds_xr = xr.Dataset.from_dataframe(df)
ds_xr

In [ ]:
ds_xr.reindex(subbasin=subbasins)

In [ ]:
import dask.dataframe as dd
dd.from_pyarrow_dataset(pyarrow_dataset)

In [ ]:
pa.fragments

In [ ]:
tmp = store.read_dynamic(train_basins[-1], start_date = '2024-01-01', end_date = '2024-02-01')
tmp

In [ ]:
for col in tmp.columns:
    print(f"{col}: {tmp[col].dtype}")

In [ ]:
for idx, grp in tmp.groupby('subbasin'):
    grp.droplevel(0,0)['gauge']['discharge'].plot()